# NSD — Processamento de Dados
Dois pipelines:
1. **COMPLETE** — todos os trials individuais, sem restrição
2. **AVG** — média dos betas por imagem única

In [ ]:
import os
import gc
import json 
import pickle
import warnings
import random
import numpy as np
import scipy.io as sio
import h5py
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter, defaultdict
from sklearn.model_selection import train_test_split
from PIL import Image

# ── Sujeitos ────────────────────────────────────────────────
SUBJECTS       = ["subj01"]
SUBJECT_INDICES = {"subj01": 0}

# ── Paths de entrada ────────────────────────────────────────
NSD_BASE         = "/home/al.pedro.alberti/Downloads/dataset/nsd"
EXPDESIGN        = os.path.join(NSD_BASE, "nsd_expdesign.mat")
STIM_INFO        = "/mnt/storage_C1/PedroAlberti/ml2/ml2_trabalhos_2026/trabalho_4/cnn_final/data_loader/auxiliar/nsd_stim_info_merged.pkl"
COCO_ANNOTATIONS = "/mnt/storage_C1/PedroAlberti/ml2/ml2_trabalhos_2026/trabalho_4/cnn_final/data_loader/auxiliar/instances_train2017.json"
COCO_IMAGES_DIR  = "/mnt/storage_C1/PedroAlberti/ml2/ml2_trabalhos_2026/trabalho_4/cnn_final/data_loader/auxiliar/train2017"

# ── Paths de saída ──────────────────────────────────────────
BASE_OUTPUT = "/home/al.pedro.alberti/Downloads/dataset/data_nsd"

OUTPUT_COMPLETE_TRAIN = os.path.join(BASE_OUTPUT, "processed/complete/train")
OUTPUT_COMPLETE_TEST  = os.path.join(BASE_OUTPUT, "processed/complete/test")


OUTPUT_AVG_TRAIN      = os.path.join(BASE_OUTPUT, "processed/AVG/train")
OUTPUT_AVG_TEST       = os.path.join(BASE_OUTPUT, "processed/AVG/test")

# ── Parâmetros ──────────────────────────────────────────────
TARGET_SHAPE = (91, 109, 91)
TEST_SIZE    = 0.15
RANDOM_STATE = 42

# ── Threshold de dominância pra rotulagem (mesmo critério do pipeline ROI) ──
# Fração mínima da ÁREA DA IMAGEM que pessoa/animal precisam ocupar pra
# "contar" no rótulo, em vez de qualquer presença incidental (ex: pessoa
# ocupando 0.4% da imagem ao fundo). Valores calibrados em process_data_ROI.ipynb.
PERSON_MIN_FRAC = 0.05
ANIMAL_MIN_FRAC = 0.02

# ── Mapeamento semântico (3 classes) ────────────────────────
SUPERCAT_MAP = {
    "person":    0,
    "animal":    1,
    "vehicle":   2,
    "indoor":    2,
    "outdoor":   2,
    "furniture": 2,
    "food":      2,
    "kitchen":   2,
    "sports":    2,
    "electronic":2,
    "appliance": 2,
    "accessory": 2,
}

NOMES_CLASSES = {0: "Pessoa", 1: "Animal", 2: "Outro"}

print("[config] Configurações carregadas.")

In [ ]:
def load_nsd_stim_info(stim_info_path: str) -> dict:
    print(f"[stim_info] Carregando: {stim_info_path}")
    with open(stim_info_path, "rb") as f:
        stim_df = pickle.load(f, encoding="latin-1")
    nsd_to_coco = dict(zip(stim_df["nsdId"], stim_df["cocoId"]))
    print(f"[stim_info] {len(nsd_to_coco):,} índices mapeados")
    return nsd_to_coco


def compute_image_areas(annotations_path: str) -> tuple:
    """
    Um passe pelas anotações COCO: soma de área por label (0/1/2) por imagem
    e área total de cada imagem. Base para rotular por fração
    de área em vez de mera presença da categoria.
    """
    with open(annotations_path) as f:
        coco_data = json.load(f)

    cat_to_label = {
        cat["id"]: SUPERCAT_MAP.get(cat["supercategory"], 2)
        for cat in coco_data["categories"]
    }
    img_area = {im["id"]: im["width"] * im["height"] for im in coco_data["images"]}

    area_by_label = defaultdict(lambda: defaultdict(float))
    for ann in coco_data["annotations"]:
        label = cat_to_label.get(ann["category_id"], 2)
        area_by_label[ann["image_id"]][label] += ann["area"]

    return area_by_label, img_area


def resolve_label(areas: dict, image_area: float,
                   person_min_frac: float = 0.0, animal_min_frac: float = 0.0) -> int:
    """
    Prioridade pessoa > animal > outro, mas pessoa/animal só "contam" se a
    área ocupar pelo menos `*_min_frac` da imagem (filtra presença incidental,
    ex: pessoa ocupando 0.4% da imagem ao fundo). 
    """
    if image_area <= 0:
        return 2
    person_area = areas.get(0, 0.0)
    animal_area = areas.get(1, 0.0)
    if person_area > 0 and (person_area / image_area) >= person_min_frac:
        return 0
    if animal_area > 0 and (animal_area / image_area) >= animal_min_frac:
        return 1
    return 2


def build_coco_label_map(annotations_path: str, person_min_frac: float = 0.0, animal_min_frac: float = 0.0) -> dict:
    """coco_id → label (0=pessoa, 1=animal, 2=outro), por fração de área."""

    print(f"[coco] Carregando anotações...")
    area_by_label, img_area = compute_image_areas(annotations_path)

    image_to_label = {
        img_id: resolve_label(areas, img_area.get(img_id, 0), person_min_frac, animal_min_frac)
        for img_id, areas in area_by_label.items()
    }
    print(f"[coco] {len(image_to_label):,} imagens catalogadas")
    return image_to_label


def build_trial_to_coco(expdesign_path: str, subject: str, nsd_to_coco: dict) -> np.ndarray:
    """Retorna array (30000,) com coco_id por trial."""

    mat       = sio.loadmat(expdesign_path)
    mo        = mat["masterordering"].flatten()
    subjectim = mat["subjectim"]
    subj_idx  = SUBJECT_INDICES[subject]

    # subjectim é 1-indexed (1..73000); nsdId no stim_info é 0-indexed (0..72999).
    # subtrai 1 pra converter o VALOR de subjectim em chave 0-indexed do nsd_to_coco.
    # Sem este -1 os rótulos ficam deslocados em 1 imagem na ordenação 73k → chance.
    nsd_indices = subjectim[subj_idx, mo - 1] - 1
    coco_ids    = np.array(
        [nsd_to_coco.get(int(i), -1) for i in nsd_indices],
        dtype=np.int32
    )

    missing = (coco_ids == -1).sum()
    print(f"[expdesign] {subject}: {len(coco_ids)} trials | "
          f"{len(np.unique(coco_ids[coco_ids>0]))} únicos | "
          f"{missing} sem match")
    return coco_ids


def build_trial_to_coco_with_nsd(expdesign_path: str, subject: str, nsd_to_coco: dict) -> tuple:
    """Retorna (nsd_ids, coco_ids) arrays (30000,)."""

    mat       = sio.loadmat(expdesign_path)
    mo        = mat["masterordering"].flatten()
    subjectim = mat["subjectim"]
    subj_idx  = SUBJECT_INDICES[subject]

    # subjectim é 1-indexed (1..73000); nsdId no stim_info é 0-indexed (0..72999).
    # subtrai 1 pra converter o VALOR de subjectim em chave 0-indexed do nsd_to_coco.
    # Sem este -1 os rótulos ficam deslocados em 1 imagem na ordenação 73k → chance.
    nsd_indices = (subjectim[subj_idx, mo - 1] - 1).astype(np.int32)
    coco_ids    = np.array(
        [nsd_to_coco.get(int(i), -1) for i in nsd_indices],
        dtype=np.int32
    )
    return nsd_indices, coco_ids


def normalize_betas(betas: np.ndarray) -> np.ndarray:
    """
    Z-score por voxel ao longo dos trials (in-place, memory-safe).
    Faz UMA cópia int16->float32 e opera in-place daí pra frente; evita as
    cópias extras de nan_to_num e de (betas-mean)/std que, na sessão inteira
    (~2GB cada), faziam o pico de RSS estourar e matar o kernel.
    """
    betas = betas.astype(np.float32)                                 
    np.nan_to_num(betas, nan=0.0, posinf=0.0, neginf=0.0, copy=False) 
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        mean = betas.mean(axis=0, keepdims=True)
        std  = betas.std(axis=0, keepdims=True)
    std = np.where(std < 1e-8, 1.0, std)
    betas -= mean   # in-place
    betas /= std    # in-place
    return betas


def resize_volume(vol: np.ndarray, target: tuple) -> torch.Tensor:
    """(X,Y,Z) → target via interpolação trilinear."""

    vol = np.nan_to_num(vol, nan=0.0, posinf=0.0, neginf=0.0)
    t   = torch.tensor(vol).unsqueeze(0).unsqueeze(0).float()
    t   = F.interpolate(t, size=target, mode="trilinear", align_corners=False)
    return t.squeeze(0).squeeze(0)


def make_split_by_coco_id(valid_indices: list, coco_ids: np.ndarray, image_to_label: dict) -> tuple:
    """Split treino/teste por coco_id único com stratify. Evita leakage."""

    coco_ids_sel  = [int(coco_ids[i]) for i in valid_indices]
    ids_map       = {cid: image_to_label.get(cid, 2) for cid in set(coco_ids_sel)}
    ids_unicos    = list(ids_map.keys())
    labels_unicos = [ids_map[cid] for cid in ids_unicos]

    train_ids, test_ids = train_test_split(
        ids_unicos,
        test_size=TEST_SIZE,
        stratify=labels_unicos,
        random_state=RANDOM_STATE
    )
    print(f"[split] {len(train_ids):,} treino | {len(test_ids):,} teste (imagens únicas)")
    return set(train_ids), set(test_ids)


def get_hdf5_files(subject: str) -> list:
    betas_dir = os.path.join(NSD_BASE, subject, "betas")
    return sorted([
        os.path.join(betas_dir, f)
        for f in os.listdir(betas_dir)
        if f.startswith("betas_session") and f.endswith(".hdf5")
    ])


def build_trial_map(hdf5_files: list) -> list:
    """Retorna lista de (hdf5_path, local_idx) para cada trial global."""

    trial_map = []
    for hdf5_path in hdf5_files:
        with h5py.File(hdf5_path, "r") as f:
            n = f["betas"].shape[0]
        for local_idx in range(n):
            trial_map.append((hdf5_path, local_idx))
    return trial_map


def plot_distribution(contador_treino: Counter, contador_teste: Counter, titulo: str = ""):
    classes = sorted(NOMES_CLASSES.keys())
    nomes   = [NOMES_CLASSES[c] for c in classes]
    vt      = [contador_treino.get(c, 0) for c in classes]
    ve      = [contador_teste.get(c, 0) for c in classes]
    tt, te  = sum(vt), sum(ve)

    x, w = np.arange(len(classes)), 0.35
    fig, ax = plt.subplots(figsize=(10, 6))
    bt = ax.bar(x - w/2, vt, w, label=f"Treino (n={tt:,})", color="#2A9D8F", edgecolor="black", alpha=0.85)
    be = ax.bar(x + w/2, ve, w, label=f"Teste  (n={te:,})", color="#E76F51", edgecolor="black", alpha=0.85)

    ax.set_title(f"Distribuição — {titulo}\nTotal: {tt+te:,}", fontsize=13, fontweight="bold")
    ax.set_xticks(x); ax.set_xticklabels(nomes)
    ax.set_ylabel("Amostras"); ax.legend()
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    for spine in ["top","right"]: ax.spines[spine].set_visible(False)

    mv = max(max(vt), max(ve))
    for bar, total in [(bt, tt), (be, te)]:
        for b in bar:
            h = b.get_height()
            if h > 0:
                ax.text(b.get_x() + b.get_width()/2, h + mv*0.01,
                        f"{int(h):,}\n({h/total*100:.1f}%)",
                        ha="center", va="bottom", fontsize=8, fontweight="bold")
    plt.tight_layout(); plt.show()

print("[utils] Funções auxiliares carregadas.")

---
## Pipeline 1 — COMPLETE
Todos os trials válidos, sem restrição de balanceamento.

In [ ]:
# ============================================================
# PIPELINE 1 — COMPLETE
# Todos os trials com label válido, trial individual
# ============================================================

nsd_to_coco    = load_nsd_stim_info(STIM_INFO)
image_to_label = build_coco_label_map(
    COCO_ANNOTATIONS, person_min_frac=PERSON_MIN_FRAC, animal_min_frac=ANIMAL_MIN_FRAC
)

contador_treino = Counter()
contador_teste  = Counter()

for subject in SUBJECTS:

    print(f"\n{'='*65}")
    print(f"[COMPLETE] {subject}")
    print(f"{'='*65}")

    out_treino = os.path.join(OUTPUT_COMPLETE_TRAIN, subject)
    out_teste  = os.path.join(OUTPUT_COMPLETE_TEST,  subject)
    os.makedirs(out_treino, exist_ok=True)
    os.makedirs(out_teste,  exist_ok=True)

    coco_ids   = build_trial_to_coco(EXPDESIGN, subject, nsd_to_coco)
    hdf5_files = get_hdf5_files(subject)
    trial_map  = build_trial_map(hdf5_files)
    print(f"[betas] {len(hdf5_files)} sessões | {len(trial_map):,} trials totais")

    # todos os trials com label válido
    valid_indices = [
        i for i in range(min(len(trial_map), len(coco_ids)))
        if int(coco_ids[i]) in image_to_label
    ]
    print(f"[filter] {len(valid_indices):,} trials válidos")

    train_ids, test_ids = make_split_by_coco_id(valid_indices, coco_ids, image_to_label)

    # agrupa por sessão para carregar cada HDF5 apenas uma vez
    by_session = defaultdict(list)
    for global_idx in valid_indices:
        hdf5_path, local_idx = trial_map[global_idx]
        by_session[hdf5_path].append((global_idx, local_idx))

    cont_treino = cont_teste = trial_num = 0

    for hdf5_path, trials_in_session in sorted(by_session.items()):
        sess_name = Path(hdf5_path).stem
        print(f"  [session] {sess_name} — {len(trials_in_session)} trials")

        with h5py.File(hdf5_path, "r") as f:
            betas_sess = f["betas"][:]

        betas_sess = normalize_betas(betas_sess)

        for global_idx, local_idx in sorted(trials_in_session, key=lambda x: x[1]):
            coco_id = int(coco_ids[global_idx])
            y       = image_to_label[coco_id]

            vol       = betas_sess[local_idx]
            vol_res   = resize_volume(vol, TARGET_SHAPE)
            X         = vol_res.unsqueeze(0)  # (1, 91, 109, 91)

            if coco_id in train_ids:
                destino = os.path.join(out_treino, f"trial_{trial_num:05d}.p")
                contador_treino[y] += 1
                cont_treino += 1
            else:
                destino = os.path.join(out_teste, f"trial_{trial_num:05d}.p")
                contador_teste[y] += 1
                cont_teste += 1

            torch.save(
                {"X": X, "y": y, "coco_id": coco_id, "subject": subject},
                destino
            )
            trial_num += 1

        del betas_sess
        gc.collect()

    print(f"\n[done] {subject}: {cont_treino:,} treino | {cont_teste:,} teste")

print("\n" + "="*65)
print("DISTRIBUIÇÃO FINAL — COMPLETE")
print("="*65)
for c in sorted(NOMES_CLASSES):
    print(f"  {NOMES_CLASSES[c]}: treino={contador_treino[c]:,} | teste={contador_teste[c]:,}")

plot_distribution(contador_treino, contador_teste, titulo="COMPLETE")

---
## Pipeline 2 — AVG
Média dos betas por imagem única (todas as repetições).

In [ ]:
# NOTA de memória: o volume completo (~699k voxels) torna o AVG pesado.

nsd_to_coco    = load_nsd_stim_info(STIM_INFO)
image_to_label = build_coco_label_map(
    COCO_ANNOTATIONS, person_min_frac=PERSON_MIN_FRAC, animal_min_frac=ANIMAL_MIN_FRAC
)

contador_treino = Counter()
contador_teste  = Counter()

for subject in SUBJECTS:

    print(f"\n{'='*65}")
    print(f"[AVG] {subject}")
    print(f"{'='*65}")

    out_treino = os.path.join(OUTPUT_AVG_TRAIN, subject)
    out_teste  = os.path.join(OUTPUT_AVG_TEST,  subject)
    os.makedirs(out_treino, exist_ok=True)
    os.makedirs(out_teste,  exist_ok=True)

    nsd_ids, coco_ids = build_trial_to_coco_with_nsd(EXPDESIGN, subject, nsd_to_coco)

    hdf5_files = get_hdf5_files(subject)
    trial_map  = build_trial_map(hdf5_files)

    # pega shape do volume
    with h5py.File(hdf5_files[0], "r") as f:
        vol_shape = f["betas"].shape[1:]
    print(f"[betas] {len(hdf5_files)} sessões | vol_shape={vol_shape}")

    valid_indices = [
        i for i in range(min(len(trial_map), len(coco_ids)))
        if int(coco_ids[i]) in image_to_label
    ]
    print(f"[filter] {len(valid_indices):,} trials válidos")

    # split por coco_id
    train_coco_ids, test_coco_ids = make_split_by_coco_id(
        valid_indices, coco_ids, image_to_label
    )

    # mapeia nsd_id → treino/teste
    train_nsd_set, test_nsd_set = set(), set()
    for idx in valid_indices:
        nid = int(nsd_ids[idx])
        cid = int(coco_ids[idx])
        if   cid in train_coco_ids: train_nsd_set.add(nid)
        elif cid in test_coco_ids:  test_nsd_set.add(nid)

    train_nsd_list = sorted(train_nsd_set)
    test_nsd_list  = sorted(test_nsd_set)
    print(f"[split] {len(train_nsd_list):,} nsd_ids treino | {len(test_nsd_list):,} teste")

    nsd_to_train_idx = {nid: i for i, nid in enumerate(train_nsd_list)}
    nsd_to_test_idx  = {nid: i for i, nid in enumerate(test_nsd_list)}

    train_coco_arr = np.array([nsd_to_coco[nid] for nid in train_nsd_list], dtype=np.int32)
    test_coco_arr  = np.array([nsd_to_coco[nid] for nid in test_nsd_list],  dtype=np.int32)

    train_out_path = os.path.join(out_treino, "averaged_betas.hdf5")
    test_out_path  = os.path.join(out_teste,  "averaged_betas.hdf5")

    with h5py.File(train_out_path, "w") as f:
        f.create_dataset("betas",    shape=(len(train_nsd_list), *vol_shape), dtype=np.float32)
        f.create_dataset("nsd_ids",  data=np.array(train_nsd_list, dtype=np.int32))
        f.create_dataset("coco_ids", data=train_coco_arr)
        f.create_dataset("labels",   data=np.array(
            [image_to_label[cid] for cid in train_coco_arr], dtype=np.int32))

    with h5py.File(test_out_path, "w") as f:
        f.create_dataset("betas",    shape=(len(test_nsd_list), *vol_shape), dtype=np.float32)
        f.create_dataset("nsd_ids",  data=np.array(test_nsd_list, dtype=np.int32))
        f.create_dataset("coco_ids", data=test_coco_arr)
        f.create_dataset("labels",   data=np.array(
            [image_to_label[cid] for cid in test_coco_arr], dtype=np.int32))

    # garante que os handles fecham mesmo se algo falhar no meio
    # (senão o arquivo fica aberto na sessão e o próximo "w" falha ao truncar)
    f_train = h5py.File(train_out_path, "r+")
    f_test  = h5py.File(test_out_path,  "r+")

    try:
        # ── FASE 1: acumula betas brutos ──────────────────────
        print("  [phase 1] Acumulando betas brutos...")

        train_counts = np.zeros(len(train_nsd_list), dtype=np.int32)
        test_counts  = np.zeros(len(test_nsd_list),  dtype=np.int32)

        by_session = defaultdict(list)
        for global_idx in valid_indices:
            hdf5_path, local_idx = trial_map[global_idx]
            by_session[hdf5_path].append((global_idx, local_idx))

        for hdf5_path, trials_in_session in sorted(by_session.items()):
            sess_name = Path(hdf5_path).stem
            print(f"    [session] {sess_name} ({len(trials_in_session)} trials)", flush=True)

            # mantém a sessão em int16 (~1GB); converte por trial (2.8MB)
            with h5py.File(hdf5_path, "r") as f:
                betas_sess = f["betas"][:]

            for global_idx, local_idx in sorted(trials_in_session, key=lambda x: x[1]):
                nsd_id = int(nsd_ids[global_idx])
                vol    = betas_sess[local_idx].astype(np.float32)
                vol    = np.nan_to_num(vol, nan=0.0, posinf=0.0, neginf=0.0)

                if nsd_id in nsd_to_train_idx:
                    idx = nsd_to_train_idx[nsd_id]
                    f_train["betas"][idx] += vol
                    train_counts[idx]     += 1
                elif nsd_id in nsd_to_test_idx:
                    idx = nsd_to_test_idx[nsd_id]
                    f_test["betas"][idx]  += vol
                    test_counts[idx]      += 1

            del betas_sess
            gc.collect()
            # converte páginas sujas em limpas (recuperáveis) — evita estourar a RAM
            f_train.flush(); f_test.flush()
            print(f"    [done] sessão concluída", flush=True)

        # ── FASE 2: divide pela contagem → média ──────────────
        print("  [phase 2] Calculando médias...")
        for i in range(len(train_nsd_list)):
            if train_counts[i] > 0:
                f_train["betas"][i] /= train_counts[i]
        for i in range(len(test_nsd_list)):
            if test_counts[i] > 0:
                f_test["betas"][i] /= test_counts[i]
        f_train.flush(); f_test.flush()

        reps_treino = train_counts
        print(f"    Repetições por nsd_id (treino) — média: {reps_treino.mean():.2f} | "
              f"min: {reps_treino.min()} | max: {reps_treino.max()}")

        # ── FASE 3: z-score com stats do TREINO (sem leakage) ─
        print("  [phase 3] Normalizando (z-score, stats do treino)...")

        # Welford online para evitar carregar tudo na RAM
        mean_v = np.zeros(vol_shape, dtype=np.float64)
        M2_v   = np.zeros(vol_shape, dtype=np.float64)
        n_v    = 0

        for i in range(len(train_nsd_list)):
            vol   = f_train["betas"][i].astype(np.float64)
            n_v  += 1
            delta = vol - mean_v
            mean_v += delta / n_v
            M2_v   += delta * (vol - mean_v)

        std_v = np.sqrt(M2_v / n_v)
        std_v = np.where(std_v < 1e-8, 1.0, std_v)

        # aplica no treino
        for i in range(len(train_nsd_list)):
            f_train["betas"][i] = (f_train["betas"][i] - mean_v) / std_v
            if i % 500 == 0:
                f_train.flush()
        f_train.flush()

        # aplica AS MESMAS stats no teste
        for i in range(len(test_nsd_list)):
            f_test["betas"][i] = (f_test["betas"][i] - mean_v) / std_v
            if i % 500 == 0:
                f_test.flush()
        f_test.flush()
    finally:
        f_train.close()
        f_test.close()

    # atualiza contadores
    for cid in train_coco_arr:
        contador_treino[image_to_label[int(cid)]] += 1
    for cid in test_coco_arr:
        contador_teste[image_to_label[int(cid)]] += 1

    print(f"[done] {subject}: {len(train_nsd_list):,} treino | {len(test_nsd_list):,} teste")
    print(f"       Salvo em {train_out_path}")

print("\n" + "="*65)
print("DISTRIBUIÇÃO FINAL — AVG (imagens únicas)")
print("="*65)
for c in sorted(NOMES_CLASSES):
    print(f"  {NOMES_CLASSES[c]}: treino={contador_treino[c]:,} | teste={contador_teste[c]:,}")

plot_distribution(contador_treino, contador_teste, titulo="AVG")